In [0]:
dbutils.library.restartPython()

In [0]:
# --- Configuration ---
# bu_filter = "GRMAN-GROUP RISK MANAGEMENT"
# bu_filter = 'GRC&R-GROUP CLAIMS & RECOVERIES'
# bu_filter='GMC-GROUP MARKETING & COMMUNICATION'
# bu_filter='GBUND-GROUP BUYER UNDERWRITING'
# bu_filter='AUDIT-INTERNAL AUDIT'
# bu_filter='CRESP-CREDIT SPECIALTIES'
# bu_filter='DSBUS-DUTCH STATE BUSINESS'
# bu_filter='HR-GROUP HUMAN RESOURCES'
# bu_filter='ICP00-ICP'
# bu_filter='INWRE-INWARD REINSURANCE'
# bu_filter='S&CD-STRATEGY & CORP. DEVELOPMENT'
# bu_filter= 'SURET-SURETY'
# bu_filter='ITS-INFORMATION TECHNOLOGY SERVICES'
# bu_filter='COMNA-NORTH AMERICA'
# bu_filter='COMNN-NETHERLANDS'
# bu_filter='COMON-OCEANIA'
# bu_filter='COMSE-BELGIUM, LUXEMBOURG'
# bu_filter='COMSPB-SPAIN, PORTUGAL, BRAZIL'
# bu_filter='COMUI-UK & IRELAND'
# bu_filter='GLOBA-GLOBAL'
# bu_filter='COMAS-ASIA'
# bu_filter='CREDIT INSURANCE'
# bu_filter='FRANC-FRANCE'
# bu_filter='ITALY-ITALY'
# bu_filter='RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE'
# bu_filter='RISK2-RS2-BELGIUM, LUX, FRANCE & ITA'
# bu_filter='RISK3-RS3-NLD, APAC & AIM'
# bu_filter='RISK4-RS4-NORTH EUROPE, CIS & ACS'
# bu_filter='RISK4-RS4-NORTH EUROPE, CIS & SP'
# bu_filter='RISK5-RS5-NAFTA'
# bu_filter='GFO-GROUP FINANCE OPERATIONS'
# bu_filter='GFC-GROUP FINANCE AND CONTROL'
# bu_filter='COFIN-CORPORATE FINANCE & TAX'
# bu_filter='FINPM-FINANCE PROGRAM MANAGEMENT'
excluded_users_list = [
    'ESBWIL1',
    'GBAPAD1',
    'NLKLAX1',
    'NLNJAI1',
    'NLKKAL1',
    'NLJPLE1',
    'NLMBHA2'
]
excluded_users_sql = ",".join([f"'{user}'" for user in excluded_users_list])

# Widget parameters for SQL cells to reference via ${...}
dbutils.widgets.text("bu_filter", bu_filter, "Business Unit Filter")
dbutils.widgets.text("excluded_users_sql", excluded_users_sql, "Excluded Users SQL")

In [0]:
%sql
SELECT
  DISTINCT UP.BU, UP.UserName, up.DisplayName
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
  ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
  AND UPPER(UP.BU) = UPPER('${bu_filter}')
  AND UP.UserName NOT IN (${excluded_users_sql})
  AND UA.WEBI_Doc_ID is not null
  -- AND cms.Document_Id NOT IN (SELECT DISTINCT Document_Id FROM Owned_Reports)


In [0]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

# --- Query 1: Document access vs ownership ---
pdf = spark.sql(f"""
  WITH Owned_Reports AS (
    SELECT DISTINCT
      cms.Document_Id,
      CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(cms.created)) = 'ADMINISTRATOR'
               THEN cms.lastAuthor
               ELSE cms.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
      ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) = UPPER('{bu_filter}')
      AND UP.UserName NOT IN ({excluded_users_sql})
  )

  SELECT
    'Only Access' AS category,
    COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UPPER(UP.BU) = UPPER('{bu_filter}')
    AND UP.UserName NOT IN ({excluded_users_sql})
    AND cms.Document_Id NOT IN (SELECT DISTINCT Document_Id FROM Owned_Reports)

  UNION ALL

  SELECT
    'Access Owned' AS category,
    COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UPPER(UP.BU) = UPPER('{bu_filter}')
    AND UP.UserName NOT IN ({excluded_users_sql})
    AND cms.Document_Id IN (SELECT DISTINCT Document_Id FROM Owned_Reports)
""").toPandas()

# --- Query 2: Owned reports - normal vs scheduled ---
pdf2 = spark.sql(f"""
  WITH Owned_Reports AS (
    SELECT DISTINCT
      cms.Document_Id,
      CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled,
      S1.Active_Schedule as schedule_active
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(cms.created)) = 'ADMINISTRATOR'
               THEN cms.lastAuthor
               ELSE cms.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
      ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) = UPPER('{bu_filter}') 
      AND UP.UserName NOT IN ({excluded_users_sql})
  )

  SELECT
    'Total owned normal reports' AS category,
    COUNT(DISTINCT Document_Id) AS Document_cnt
  FROM Owned_Reports
  WHERE scheduled = 'N'
  or( scheduled ='Y' and schedule_active = false)

  UNION ALL

  SELECT
    'Total owned scheduled reports' AS category,
    COUNT(DISTINCT Document_Id) AS Document_cnt
  FROM Owned_Reports
  WHERE scheduled = 'Y'
  AND  schedule_active = true
""").toPandas()

# --- Combined Chart: Two stacked bars side by side (Navy Theme) ---
access_owned = pdf.loc[pdf['category'] == 'Access Owned', 'Document_cnt'].values[0]
only_access = pdf.loc[pdf['category'] == 'Only Access', 'Document_cnt'].values[0]
normal = pdf2.loc[pdf2['category'] == 'Total owned normal reports', 'Document_cnt'].values[0]
scheduled = pdf2.loc[pdf2['category'] == 'Total owned scheduled reports', 'Document_cnt'].values[0]

# Print total reports in BO Audit log bar
print(f"Total reports in BO Audit log bar: {access_owned + only_access:,}")

fig, ax = plt.subplots(figsize=(6, 7))

width = 0.25  # Narrower bars
positions = [0, 0.5]  # Closer together

# Proportional threshold and offset
y_max = max(access_owned + only_access, normal + scheduled)
min_height_for_inside = y_max * 0.08  # 8% of chart height
label_offset = y_max * 0.02  # 2% gap above bar

# Bar 1: Access distribution
ax.bar(positions[0], access_owned, width=width, color='#336699', label='Reports Owned')
ax.bar(positions[0], only_access, width=width, bottom=access_owned, color='#FABE6E', label='Reports Outside BU')

# Bar 1 labels
total_1 = access_owned + only_access
if access_owned < min_height_for_inside and only_access < min_height_for_inside:
    offset = total_1 + label_offset
    ax.text(positions[0] - 0.06, offset, f'{access_owned:,}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#336699')
    ax.text(positions[0], offset, '|', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#336699')
    ax.text(positions[0] + 0.06, offset, f'{only_access:,}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#FABE6E')
elif access_owned < min_height_for_inside:
    ax.text(positions[0], total_1 + label_offset, f'{access_owned:,}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#336699')
    ax.text(positions[0], access_owned + only_access / 2, f'{only_access:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='black')
elif only_access < min_height_for_inside:
    ax.text(positions[0], access_owned / 2, f'{access_owned:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='white')
    ax.text(positions[0], total_1 + label_offset, f'{only_access:,}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#FABE6E')
else:
    ax.text(positions[0], access_owned / 2, f'{access_owned:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='white')
    ax.text(positions[0], access_owned + only_access / 2, f'{only_access:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='black')

# Bar 2: Owned reports breakdown
ax.bar(positions[1], normal, width=width, color='#4D7DAF', label='Owned Reports')
ax.bar(positions[1], scheduled, width=width, bottom=normal, color='#FFD700', label='Owned Reports with Schedule')

# Bar 2 labels
total_2 = normal + scheduled
if normal < min_height_for_inside and scheduled < min_height_for_inside:
    offset = total_2 + label_offset
    ax.text(positions[1] - 0.06, offset, f'{normal:,}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#4D7DAF')
    ax.text(positions[1], offset, '|', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#336699')
    ax.text(positions[1] + 0.06, offset, f'{scheduled:,}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#FFD700')
elif normal < min_height_for_inside:
    ax.text(positions[1], total_2 + label_offset, f'{normal:,}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#4D7DAF')
    ax.text(positions[1], normal + scheduled / 2, f'{scheduled:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='black')
elif scheduled < min_height_for_inside:
    ax.text(positions[1], normal / 2, f'{normal:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='white')
    ax.text(positions[1], total_2 + label_offset, f'{scheduled:,}', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#FFD700')
else:
    ax.text(positions[1], normal / 2, f'{normal:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='white')
    ax.text(positions[1], normal + scheduled / 2, f'{scheduled:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='black')

ax.set_ylim(0, y_max * 1.15)
ax.set_xticks(positions)
ax.set_xticklabels(['BO AUDIT Log\nAccessd Reports', 'BO Reports INVENTORY\nWith Schedules'])
ax.set_ylabel('Reports Count')
ax.set_title(f'{bu_filter} - Overview')
ax.legend(title='Category', fontsize=8, title_fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Query: Accessed additionally vs Owned by cluster (aligned with Cell 3) ---
pdf3 = spark.sql(f"""
  WITH owned AS (
    SELECT DISTINCT co.cluster, CAST(cms.Document_Id AS STRING) AS Doc_ID
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(CMS.Document_Id AS STRING) = co.Doc_ID
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) = UPPER('{bu_filter}')
      AND UP.UserName NOT IN ({excluded_users_sql})
      AND CAST(CMS.Document_Id AS STRING) IN (
        SELECT DISTINCT CAST(UA.WEBI_Doc_ID AS STRING)
        FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
        INNER JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP2
          ON UPPER(TRIM(UA.UserName)) = UPPER(TRIM(UP2.UserName))
        WHERE UPPER(UP2.BU) = UPPER('{bu_filter}')
          AND UP2.UserName NOT IN ({excluded_users_sql})
      )
  )

  SELECT 'Accessed additionaly' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_id) AS reports
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
  WHERE Doc_id NOT IN (SELECT Doc_ID FROM owned)
    AND UPPER(BU) = UPPER('{bu_filter}')
  GROUP BY cluster

  UNION ALL

  SELECT 'Owned' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_ID) AS reports
  FROM owned
  GROUP BY cluster
  ORDER BY category ASC, cluster ASC
""").toPandas()

pivot = pdf3.pivot(index='cluster', columns='category', values='reports').fillna(0)

def sort_key(x):
    try:
        return (0, int(x))
    except ValueError:
        return (1, 0)

pivot = pivot.loc[sorted(pivot.index, key=sort_key)]
clusters = pivot.index.tolist()
x = np.arange(len(clusters)) * 0.7

owned_vals = pivot.get('Owned', pd.Series([0]*len(clusters))).values
accessed_vals = pivot.get('Accessed additionaly', pd.Series([0]*len(clusters))).values

fig, ax = plt.subplots(figsize=(8, 6))

width = 0.45
bars_owned = ax.bar(x, owned_vals, width=width, color='#336699', label='Owned')
bars_accessed = ax.bar(x, accessed_vals, width=width, bottom=owned_vals, color='#FABE6E', label='Accessed additionaly')

y_max = max(owned_vals + accessed_vals)
label_offset = y_max * 0.02
min_height_for_inside = y_max * 0.08

for i, (o, a) in enumerate(zip(owned_vals, accessed_vals)):
    total_height = o + a
    if o > 0:
        if o < min_height_for_inside:
            if a > 0 and a < min_height_for_inside:
                offset = total_height + label_offset
                ax.text(x[i] - 0.1, offset, f'{int(o):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#336699')
                ax.text(x[i], offset, '|', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#336699')
                ax.text(x[i] + 0.1, offset, f'{int(a):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#FABE6E')
            elif a > 0:
                ax.text(x[i], total_height + label_offset, f'{int(o):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#336699')
                ax.text(x[i], o + a / 2, f'{int(a):,}', ha='center', va='center', fontsize=9, fontweight='bold', color='black')
            else:
                ax.text(x[i], total_height + label_offset, f'{int(o):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#336699')
        else:
            ax.text(x[i], o / 2, f'{int(o):,}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
            if a > 0:
                if a < min_height_for_inside:
                    ax.text(x[i], total_height + label_offset, f'{int(a):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#FABE6E')
                else:
                    ax.text(x[i], o + a / 2, f'{int(a):,}', ha='center', va='center', fontsize=9, fontweight='bold', color='black')
    elif a > 0:
        if a < min_height_for_inside:
            ax.text(x[i], a + label_offset, f'{int(a):,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#FABE6E')
        else:
            ax.text(x[i], a / 2, f'{int(a):,}', ha='center', va='center', fontsize=9, fontweight='bold', color='black')

ax.set_ylim(0, y_max * 1.15)
ax.set_xlabel('Cluster')
ax.set_ylabel('Reports Count')
ax.set_title(f'{bu_filter} - Reports by Cluster')
ax.set_xticks(x)
ax.set_xticklabels(clusters)
ax.legend(title='Category', fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# --- Query: Top Editors in BU with cluster detail ---
pdf_editors = spark.sql(f"""
  WITH eventCategorized AS (
    SELECT
      ua1.*,
      COALESCE(CAST(CAST(co.cluster AS INT) AS STRING), 'UNCLUSTERED') AS Report_Cluster,
      CASE
        WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
        ELSE 'Viewer_events'
      END AS usage_category
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
    LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
      ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(ua1.WEBI_Doc_ID AS STRING) = co.Doc_ID
    WHERE UPPER(TRIM(CMS.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND CMS.Document_Id IS NOT NULL
  )

  SELECT
    ec.UserName AS User_ID,
    up1.DisplayName AS User_Name,
    up1.BU AS User_BU,
    up1.Title AS User_Role,
    ec.Report_Cluster,
    COUNT(DISTINCT CASE WHEN usage_category = 'Editor_events' THEN WEBI_Doc_ID END) AS Editor_report_cnt,
    sum(ec.Events) as Interactions_cnt
  FROM eventCategorized AS ec
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON ec.UserName = up1.UserName
  WHERE UPPER(up1.BU) = UPPER('{bu_filter}')
    AND ec.UserName NOT IN ({excluded_users_sql})
  GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title, ec.Report_Cluster
  HAVING COUNT(DISTINCT CASE WHEN usage_category = 'Editor_events' THEN WEBI_Doc_ID END) > 0
""").toPandas()

editor_totals = pdf_editors.groupby(['User_ID', 'User_Name', 'User_Role']).agg(
    Editor_report_cnt=('Editor_report_cnt', 'sum'),
    Interactions_cnt=('Interactions_cnt', 'sum')
).reset_index()
editor_totals = editor_totals.sort_values('Editor_report_cnt', ascending=False).head(7)
top_editor_ids = editor_totals['User_ID'].tolist()

pdf_editors_top = pdf_editors[pdf_editors['User_ID'].isin(top_editor_ids)].copy()

editor_ids_sql = ",".join([f"'{uid}'" for uid in top_editor_ids])
pdf_viewers = spark.sql(f"""
  WITH eventCategorized AS (
    SELECT
      ua1.*,
      COALESCE(CAST(CAST(co.cluster AS INT) AS STRING), 'UNCLUSTERED') AS Report_Cluster,
      CASE
        WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
        ELSE 'Viewer_events'
      END AS usage_category
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
    LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
      ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(ua1.WEBI_Doc_ID AS STRING) = co.Doc_ID
    WHERE UPPER(TRIM(CMS.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND CMS.Document_Id IS NOT NULL
  )

  SELECT
    ec.UserName AS User_ID,
    up1.DisplayName AS User_Name,
    up1.BU AS User_BU,
    up1.Title AS User_Role,
    ec.Report_Cluster,
    COUNT(DISTINCT CASE WHEN usage_category = 'Viewer_events' THEN WEBI_Doc_ID END) AS Viewer_report_cnt,
    sum(ec.Events) as Interactions_cnt
  FROM eventCategorized AS ec
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON ec.UserName = up1.UserName
  WHERE UPPER(up1.BU) = UPPER('{bu_filter}')
    AND up1.UserName NOT IN ({editor_ids_sql})
    AND ec.UserName NOT IN ({excluded_users_sql})
  GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title, ec.Report_Cluster
  HAVING COUNT(DISTINCT CASE WHEN usage_category = 'Viewer_events' THEN WEBI_Doc_ID END) > 0
""").toPandas()

viewer_totals = pdf_viewers.groupby(['User_ID', 'User_Name', 'User_Role']).agg(
    Viewer_report_cnt=('Viewer_report_cnt', 'sum'),
    Interactions_cnt=('Interactions_cnt', 'sum')
).reset_index()
viewer_totals = viewer_totals.sort_values('Viewer_report_cnt', ascending=False).head(7)
top_viewer_ids = viewer_totals['User_ID'].tolist()

pdf_viewers_top = pdf_viewers[pdf_viewers['User_ID'].isin(top_viewer_ids)].copy()

def generate_palette(cmap_name, n=12):
    """Generate a smooth gradient palette from a matplotlib colormap."""
    cmap = plt.cm.get_cmap(cmap_name, n + 2)
    palette = {}
    for i in range(n):
        rgba = cmap(0.15 + (i / (n - 1)) * 0.77)
        palette[str(i)] = '#{:02x}{:02x}{:02x}'.format(int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255))
    palette['UNCLUSTERED'] = '#D4D4D4'
    return palette

def get_text_color(hex_color):
    """Return white for dark backgrounds, black for light backgrounds based on luminance."""
    hex_color = hex_color.lstrip('#')
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    luminance = (0.299 * r + 0.587 * g + 0.114 * b) / 255
    return 'white' if luminance < 0.5 else 'black'

blue_palette = generate_palette('Blues', 12)
orange_palette = generate_palette('Oranges', 12)

def cluster_sort_key(c):
    try:
        return (0, int(c))
    except ValueError:
        return (1, 0)

def plot_cluster_barh(pdf, user_col, count_col, title, top_ids, totals_df, palette):
    user_order = totals_df.sort_values(count_col, ascending=True)['User_ID'].tolist()
    user_names = totals_df.set_index('User_ID')['User_Name'].to_dict()
    
    pivot = pdf.pivot_table(index='User_ID', columns='Report_Cluster', values=count_col, aggfunc='sum', fill_value=0)
    pivot = pivot.reindex(user_order)
    
    all_clusters = sorted(pivot.columns.tolist(), key=cluster_sort_key)
    pivot = pivot[all_clusters]
    
    numeric_clusters = [c for c in all_clusters if c != 'UNCLUSTERED']
    user_top2 = {}
    for uid in user_order:
        user_vals = pivot.loc[uid, numeric_clusters].sort_values(ascending=False)
        user_top2[uid] = user_vals.head(2).index[user_vals.head(2) > 0].tolist()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    y_pos = np.arange(len(user_order))
    bar_height = 0.5
    
    left = np.zeros(len(user_order))
    bar_positions = {}
    for cluster in all_clusters:
        vals = pivot[cluster].values
        color = palette.get(cluster, '#D4D4D4')
        ax.barh(y_pos, vals, left=left, height=bar_height, color=color, edgecolor='white', linewidth=0.5, label=cluster)
        bar_positions[cluster] = (left.copy(), vals, color)
        left += vals
    
    for i, uid in enumerate(user_order):
        for cluster in user_top2.get(uid, []):
            lefts, vals, color = bar_positions[cluster]
            v = vals[i]
            if v > 0:
                txt_color = get_text_color(color)
                ax.text(lefts[i] + v / 2, y_pos[i], f'{int(v)}', ha='center', va='center',
                        fontsize=7, fontweight='bold', color=txt_color)
    
    x_max = max(left) if len(left) > 0 and max(left) > 0 else 1
    label_gap = x_max * 0.02
    for i, uid in enumerate(user_order):
        total = int(left[i])
        ax.text(left[i] + label_gap, y_pos[i], f'{total:,}', va='center', fontsize=9, fontweight='bold', color='#333333')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels([user_names.get(uid, uid) or uid for uid in user_order], fontsize=10)
    ax.set_xlabel('Distinct Reports')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(title='Cluster', fontsize=8, loc='lower right', ncol=4, framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_cluster_barh(pdf_editors_top, 'User_ID', 'Editor_report_cnt',
                  f'{bu_filter} - Top 7 Editors by Report Count (by Cluster)',
                  top_editor_ids, editor_totals, blue_palette)

plot_cluster_barh(pdf_viewers_top, 'User_ID', 'Viewer_report_cnt',
                  f'{bu_filter} - Top 7 Viewers by Report Count (by Cluster)',
                  top_viewer_ids, viewer_totals, orange_palette)
# 
editor_totals_disp = editor_totals.rename(columns={'User_Name': 'Name', 'User_Role': 'Role', 'Interactions_cnt': 'Interactions', 'Editor_report_cnt': 'Reports'})
editor_totals_disp.insert(0, 'Category', 'Editor')
editor_totals_disp = editor_totals_disp.drop(columns=['User_ID'])

viewer_totals_disp = viewer_totals.rename(columns={'User_Name': 'Name', 'User_Role': 'Role', 'Interactions_cnt': 'Interactions', 'Viewer_report_cnt': 'Reports'})
viewer_totals_disp.insert(0, 'Category', 'Viewer')
viewer_totals_disp = viewer_totals_disp.drop(columns=['User_ID'])

pdf_combined = pd.concat([editor_totals_disp, viewer_totals_disp], ignore_index=True)

cols = [c for c in pdf_combined.columns if c != 'Reports'] + ['Reports']
pdf_combined = pdf_combined[cols]

for col in pdf_combined.select_dtypes(include='number').columns:
    pdf_combined[col] = pdf_combined[col].apply(lambda x: f'{int(x):,}')

display(pdf_combined)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Query: Schedule Report Owners ---
pdf_sched_owners = spark.sql(f"""
WITH labled_schedule AS (
  SELECT
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email multiple users'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Drive'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
  ON CAST(sd1.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE UPPER(up1.BU) = UPPER('{bu_filter}')
  AND sd2.Active_Schedule = TRUE
  AND up1.UserName NOT IN ({excluded_users_sql})
)
SELECT
  UP.DisplayName,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END AS Category,
  COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(
      CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
           THEN CMS.lastAuthor
           ELSE CMS.created
      END
  )) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
  ON CAST(cms.Document_Id AS STRING) = sd2.document_id
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
  ON sd3.document_id = sd2.document_id
LEFT JOIN labled_schedule
  ON sd2.document_id = labled_schedule.document_id
WHERE UPPER(UP.BU) = UPPER('{bu_filter}')
  AND sd2.document_id IS NOT NULL
  AND sd3.Active_Schedule = TRUE
  AND UP.UserName NOT IN ({excluded_users_sql})
GROUP BY
  UP.DisplayName,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END
ORDER BY Category ASC, Reports_cnt DESC
""").toPandas()

# --- Print info outside chart ---
print(f"Unique Users: {pdf_sched_owners['DisplayName'].nunique()} | DataFrame Rows: {len(pdf_sched_owners)}")

# --- Pivot for stacked bar chart ---
pivot = pdf_sched_owners.pivot(index='DisplayName', columns='Category', values='Reports_cnt').fillna(0)
owners = pivot.index
categories = pivot.columns
colors = {
    'Email self': '#5DADE2',
    'Email multiple users': '#1B4F72',
    'Shared Drive': '#F4D03F',
    'SAP BO': '#E67E22',
    'Paused': '#C0C0C0'
}

fig, ax = plt.subplots(figsize=(12, 6))
width = 0.4  # Thinner bars
bottom = pd.Series([0]*len(owners), index=owners)
for cat in categories:
    ax.bar(owners, pivot[cat], width=width, bottom=bottom, color=colors.get(cat, '#888888'), label=cat)
    bottom += pivot[cat]

ax.set_ylabel('Scheduled Reports Count')
ax.set_title(f'{bu_filter} - Scheduled Reports by Owners')
ax.legend(title='Schedule Category', fontsize=10, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- Display table ---
if not pdf_sched_owners.empty:
    display(pdf_sched_owners)

In [0]:
%sql
WITH eventCategorized AS (
  SELECT
    ua1.*,
    COALESCE(CAST(CAST(co.cluster AS INT) AS STRING), 'UNCLUSTERED') AS Report_Cluster,
    CASE
      WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
    ON CAST(ua1.WEBI_Doc_ID AS STRING) = co.Doc_ID
  WHERE CMS.Document_name NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND CMS.Document_Id IS NOT NULL
    limit 10
)

SELECT
  ec.UserName as User_ID, up1.DisplayName as User_Name, up1.BU as User_Bu, up1.Title as User_Role,
  SUM(CASE WHEN usage_category = 'Viewer_events' THEN 1 ELSE 0 END) AS Viewer_events,
  COUNT(DISTINCT CASE WHEN usage_category = 'Viewer_events' THEN ec.WEBI_Doc_ID END) AS Viewer_report_cnt
FROM eventCategorized AS ec
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
  ON ec.UserName = up1.UserName
WHERE UPPER(up1.BU) = UPPER('${bu_filter}')
-- and up1.UserName not in ('DESWAH1','DEJSCH6','DECGAS2','DEMJAN2','CZLBEC1','DEIRAT2','CHPSTU1')
GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title
ORDER BY Viewer_report_cnt DESC
limit 7

In [0]:
%sql
WITH owned AS (
    SELECT DISTINCT co.cluster, CAST(cms.Document_Id AS STRING) AS Doc_ID
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(
          CASE WHEN UPPER(TRIM(CMS.created)) = 'ADMINISTRATOR'
               THEN CMS.lastAuthor
               ELSE CMS.created
          END
      )) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis AS co
      ON CAST(CMS.Document_Id AS STRING) = co.Doc_ID
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UPPER(UP.BU) = UPPER('${bu_filter}')
      AND CAST(CMS.Document_Id AS STRING) IN (
        SELECT DISTINCT CAST(UA.WEBI_Doc_ID AS STRING)
        FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
        INNER JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP2
          ON UPPER(TRIM(UA.UserName)) = UPPER(TRIM(UP2.UserName))
        WHERE UPPER(UP2.BU) = UPPER('${bu_filter}')
      )
)

SELECT 'Accessed additionaly' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_id) AS reports
FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis 
WHERE Doc_id NOT IN (SELECT Doc_ID FROM owned)
  AND UPPER(BU) = UPPER('${bu_filter}')
GROUP BY cluster

UNION ALL

SELECT 'Owned' AS category, COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Doc_ID) AS reports
FROM owned
GROUP BY cluster
ORDER BY category ASC, cluster ASC

In [0]:
%sql
select distinct UserName from dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile
where DisplayName in ('PADDEN Andrew','JAIN Neha','KALIYAMOORTHY Kalaivani','BHAVARAJU Manikanthan','LAXMI Kiruthigaa','PLESSIS Jean du','WILKINSON Baodi')

In [0]:
%sql
    
WITH labled_schedule AS (
  SELECT
    sd1.document_id,
    CASE
      WHEN COUNT(DISTINCT sd1.sent_to) = 1
           AND MAX(UPPER(sd1.sent_to)) = MAX(UPPER(up1.EmailAddress)) THEN 'Email to self'
      ELSE 'Email to multiple'
    END AS Consumption
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS se1
    ON sd1.document_id = se1.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE UPPER(up1.BU) = UPPER('${bu_filter}')
    AND se1.Active_Schedule = TRUE
    AND sd1.destination_type = 'mail'
  GROUP BY sd1.document_id
)

-- SELECT
--   CASE
--     WHEN sd2.email_delivery = TRUE AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
--     WHEN sd2.BO_delivery=TRUE THEN 'SAP BO'
--     WHEN sd2.file_delivery = TRUE THEN 'Shared Drive'
--     WHEN labled_schedule.Consumption IS NULL AND sd2.email_delivery = TRUE THEN 'Paused'
--     ELSE NULL
--   END AS Category,
--   COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt,
--   COUNT(DISTINCT CMS.created) AS Unique_Owner_cnt,
--   ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(UP.DisplayName)), ' | ') AS Report_Owners
-- FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
-- LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
--   ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
-- LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
--   ON CAST(cms.Document_Id AS STRING) = sd2.document_id
-- LEFT JOIN labled_schedule
--   ON sd2.document_id = labled_schedule.document_id
-- WHERE UPPER(UP.BU) = UPPER('${bu_filter}')
--   AND sd2.document_id IS NOT NULL
--   AND sd2.Active_Schedule= TRUE
-- GROUP BY
--   CASE
--     WHEN sd2.email_delivery = TRUE AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
--     WHEN sd2.BO_delivery=TRUE THEN 'SAP BO'
--     WHEN sd2.file_delivery = TRUE THEN 'Shared Drive'
--     WHEN labled_schedule.Consumption IS NULL AND sd2.email_delivery = TRUE THEN 'Paused'
--     ELSE NULL
--   END
-- ORDER BY
--   CASE
--     WHEN
--       Category='SAP BO' THEN 1
--     WHEN
--       Category='Email to self' THEN 2
--     WHEN
--       Category='Email to multiple' THEN 3
--     WHEN
--      Category='Shared Drive' THEN 4
--     WHEN
--       Category='Paused' THEN 5
--     ELSE 6
--   END


  SELECT
  DISTINCT
  CASE
    WHEN sd2.email_delivery = TRUE AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.BO_delivery=TRUE THEN 'SAP BO'
    WHEN sd2.file_delivery = TRUE THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL AND sd2.email_delivery = TRUE THEN 'Paused'
    ELSE NULL
  END AS Category,
  CMS.Document_Id AS Report_ID,
  CMS.Full_path as Report_foler,
  up.DisplayName AS Owner,
  sd3.file_location AS Report_delivered
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
  ON CAST(cms.Document_Id AS STRING) = sd2.document_id
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_sharedlocation as sd3
 on sd2.document_id = sd3.document_id
LEFT JOIN labled_schedule
  ON sd2.document_id = labled_schedule.document_id
WHERE UPPER(UP.BU) = UPPER('${bu_filter}')
  AND sd2.document_id IS NOT NULL
  AND sd2.Active_Schedule= TRUE

ORDER BY
  CASE
    WHEN
      Category='SAP BO' THEN 1
    WHEN
      Category='Email to self' THEN 2
    WHEN
      Category='Email to multiple' THEN 3
    WHEN
     Category='Shared Drive' THEN 4
    WHEN
      Category='Paused' THEN 5
    ELSE 6
  END

In [0]:
%sql
WITH labled_schedule AS (
  SELECT
    up1.BU,
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email others'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Drive'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
  ON CAST(sd1.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE sd2.Active_Schedule = TRUE
    AND up1.UserName NOT IN (${excluded_users_sql})
),

base AS (
  SELECT
    UP.BU AS BusinessUnit,
    UP.DisplayName AS User_name,
    CASE
      WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
      WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
      WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
      WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
      ELSE sd2.destination_type
    END AS Category,
    COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON CAST(sd3.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN labled_schedule
    ON sd2.document_id = labled_schedule.document_id
  WHERE sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
    AND UP.UserName NOT IN (${excluded_users_sql})
  GROUP BY
    UP.BU,
    UP.DisplayName,
    CASE
      WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
      WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
      WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
      WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
      ELSE sd2.destination_type
    END
),

base_with_total AS (
  SELECT * FROM base
  UNION ALL
  SELECT
    UP.BU AS BusinessUnit,
    UP.DisplayName AS User_name,
    'Total' AS Category,
    COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON sd3.document_id = sd2.document_id
  LEFT JOIN labled_schedule
    ON sd2.document_id = labled_schedule.document_id
  WHERE sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
    AND UP.UserName NOT IN (${excluded_users_sql})
  GROUP BY UP.BU, UP.DisplayName
)

SELECT *
FROM base_with_total
PIVOT (
  SUM(Reports_cnt)
  FOR Category IN (
    'SAP BO' AS SAP_BO,
    'Email self' AS Email_self,
    'Email others' AS Email_others,
    'Shared Drive' AS Shared_Drive,
    'Paused' AS Paused,
    'Total' AS Total
  )
)
ORDER BY BusinessUnit, User_name